In [1]:
!nvidia-smi

/bin/bash: nvidia-smi: command not found


# Gradient-Speed Diagnostic: float32 + use_remat=True + TRUNC_NUM=6

Variant of `pymc_sampler_comparison.ipynb` that applies three changes to the JAX path and measures forward/gradient timings only (no sampling):

1. **`set_jax_precision(False)`** — float32 instead of float64 (consumer GPUs cripple FP64).
2. **`JAX_USE_REMAT = True`** — rematerialize scan body to cut reverse-mode memory.
3. **`TRUNC_NUM = 6`** — match `inference_comparison.ipynb` series-truncation length.

Goal: see whether the gradient/forward ratio drops from ~20–40× back to a healthy ~3–6×.

> **Note**: this notebook intentionally stops at the timing cell. PyMC sampling, PyTensor `Op` wrappers, and post-sampling analysis from the original notebook have been removed. Once a config looks healthy here, port it back into `pymc_sampler_comparison.ipynb` for the full NUTS run.

In [2]:
import os
import sys
import time
import datetime
import numpy as np

# Add src to path
src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pymc as pm
import pytensor
import pytensor.tensor as pt
import arviz as az
import matplotlib.pyplot as plt

from pytensor.graph.op import Op

# JAX imports
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap

from efficient_fpt.jax import set_jax_precision

set_jax_precision(False)

print(f"PyMC version: {pm.__version__}")
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

PyMC version: 5.27.0
JAX version: 0.7.2
JAX devices: [CpuDevice(id=0)]


## Configuration

In [3]:
# =====================================================================
# Configuration
# =====================================================================
DATA_PATH = "../example4_new/addm_data_20251015-163921.npz"

# Data subset (use smaller for testing, larger for real comparison)
NUM_TRIALS = 500  # Number of trials to use

# Sampling configuration
N_DRAWS = 500      # Number of posterior draws per chain
N_TUNE = 200       # Number of tuning steps
N_CHAINS = 2       # Number of chains

# Cython configuration
NUM_THREADS = 32    # OpenMP threads for Cython
PYMC_CORES = 1     # Avoid forking a multithreaded JAX process

# JAX configuration
TRUNC_NUM = 6      # Fixed truncation length used in this notebook
JAX_USE_REMAT = True

# Random seed for reproducibility
RANDOM_SEED = 42

## Load Data

In [4]:
# =====================================================================
# Load and prepare data using canonical loader
# =====================================================================
from efficient_fpt.io import load_addm_experiment

data = load_addm_experiment(DATA_PATH)
params = data["params"]
decision = data["decision_data"]
covariates = data["covariates"]

# True parameters
TRUE_PARAMS = {
    'eta': float(params["eta"]),
    'kappa': float(params["kappa"]),
    'a': float(params["a"]),
    'b': float(params["b"]),
    'x0': float(params["x0"]),
    'sigma': float(params["sigma"]),
}

# Extract data arrays
r1_full = covariates["r1_data"].astype(np.float64)
r2_full = covariates["r2_data"].astype(np.float64)
flag_full = covariates["flag_data"].astype(np.int32)
sacc_full = covariates["sacc_array_data"].astype(np.float64)
d_full = covariates["d_data"].astype(np.int32)
rt_full = decision["rt_data"].astype(np.float64)
choice_full = decision["choice_data"].astype(np.int32)

num_data_full = len(rt_full)
max_d = int(d_full.max())

# Subset to NUM_TRIALS
idx = slice(0, min(NUM_TRIALS, num_data_full))
r1_data = r1_full[idx]
r2_data = r2_full[idx]
flag_data = flag_full[idx]
sacc_data = sacc_full[idx]
d_data = d_full[idx]
rt_data = rt_full[idx]
choice_data = choice_full[idx]

num_data = len(rt_data)
sigma = TRUE_PARAMS['sigma']
M = float(np.max(rt_data))  # Max RT for constraints

print(f"Data loaded: {num_data} trials, max_d={max_d}")
print(f"Max RT: {M:.4f}")
print(f"\nTrue parameters:")
for k, v in TRUE_PARAMS.items():
    print(f"  {k}: {v}")


Data loaded: 500 trials, max_d=12
Max RT: 4.8855

True parameters:
  eta: 0.7
  kappa: 0.5
  a: 2.1
  b: 0.3
  x0: -0.2
  sigma: 1.0


## 1. Cython Implementation (Transformed-Space Metropolis)

This uses a custom PyTensor Op wrapping the Cython likelihood, but the PyMC model is written in the **same unconstrained parameterization** as the reference Bayesian notebook. Metropolis therefore samples the transformed variables rather than making random walks directly in constrained parameter space.


In [5]:
# Import Cython implementation
from efficient_fpt.cython import compute_addm_mean_nll, print_num_threads

print("Cython implementation loaded.")
print_num_threads()

Cython implementation loaded.
Number of available threads: 12


In [6]:
class LogLikeCython(Op):
    """
    PyTensor Op wrapping the Cython likelihood function.
    
    Input: theta = [eta, kappa, a, b, x0]
    Output: scalar log-likelihood
    
    NOTE: No grad() method defined = gradient-free only!
    """
    itypes = [pt.dvector]
    otypes = [pt.dscalar]

    def __init__(self, rt_data, choice_data, r1_data, r2_data, flag_data,
                 sacc_data, d_data, sigma, n_threads=8):
        self.rt_data = np.asarray(rt_data, dtype=np.float64)
        self.choice_data = np.asarray(choice_data, dtype=np.int32)
        self.r1_data = np.asarray(r1_data, dtype=np.float64)
        self.r2_data = np.asarray(r2_data, dtype=np.float64)
        self.flag_data = np.asarray(flag_data, dtype=np.int32)
        self.sacc_data = np.asarray(sacc_data, dtype=np.float64)
        self.d_data = np.asarray(d_data, dtype=np.int32)
        self.sigma = float(sigma)
        self.num_data = len(self.rt_data)
        self.n_threads = int(n_threads)

    def perform(self, node, inputs, outputs):
        (theta,) = inputs
        eta, kappa, a, b, x0 = theta

        # Compute negative log-likelihood
        nll = compute_addm_mean_nll(
            self.rt_data, self.choice_data,
            eta, kappa, self.sigma, a, b, x0,
            self.r1_data, self.r2_data, self.flag_data,
            self.sacc_data, self.d_data,
            order_mid=30, order_last=30,
            n_threads=self.n_threads,
            log_space=True,
        )
        
        # Return log-likelihood (negative of NLL)
        loglik = -self.num_data * nll
        outputs[0][0] = np.array(loglik, dtype="float64")


# Create the Cython Op
cython_loglike_op = LogLikeCython(
    rt_data=rt_data, choice_data=choice_data,
    r1_data=r1_data, r2_data=r2_data, flag_data=flag_data,
    sacc_data=sacc_data, d_data=d_data,
    sigma=sigma, n_threads=NUM_THREADS,
)

# Test it
theta_true = np.array([TRUE_PARAMS['eta'], TRUE_PARAMS['kappa'], 
                       TRUE_PARAMS['a'], TRUE_PARAMS['b'], TRUE_PARAMS['x0']])
theta_sym = pt.dvector("theta")
ll_cython = pytensor.function([theta_sym], cython_loglike_op(theta_sym))

print(f"Cython log-likelihood at true params: {ll_cython(theta_true):.4f}")

Cython log-likelihood at true params: -749.6751


## 2. JAX Implementation (Same Posterior, With Gradients) — batchscan & batchvmap

This uses the JAX-based likelihood with automatic differentiation, but keeps the **same transformed-space posterior and priors** as the Cython/PyMC model. We build both `batchscan` and `batchvmap` variants to compare their NUTS sampling performance.

In [7]:
# Import both public JAX batch NLL factories
from efficient_fpt.jax import (
    make_addm_nll_function_batchscan,
    make_addm_nll_function_batchvmap,
)

print("JAX implementation loaded (batchscan + batchvmap).")

JAX implementation loaded (batchscan + batchvmap).


In [8]:
print(
    f"Building both batchscan and batchvmap NLL closures "
    f"(trunc_num={TRUNC_NUM}, use_remat={JAX_USE_REMAT})"
)

Building both batchscan and batchvmap NLL closures (trunc_num=6, use_remat=True)


In [9]:
common_kwargs = dict(
    rt_data=rt_data,
    choice_data=choice_data,
    r1_data=r1_data,
    r2_data=r2_data,
    flag_data=flag_data,
    sacc_array_data=sacc_data,
    d_data=d_data,
    order_mid=30,
    order_last=30,
    trunc_num=TRUNC_NUM,
    log_space=True,
    use_remat=JAX_USE_REMAT,
)

jax_sum_nll_batchscan = make_addm_nll_function_batchscan(**common_kwargs)
jax_sum_nll_batchvmap = make_addm_nll_function_batchvmap(**common_kwargs)

def make_loglik_and_grad(sum_nll_fn):
    def loglik(eta, kappa, a, b, x0):
        return -sum_nll_fn(eta, kappa, sigma, a, b, x0)
    return jit(loglik), jit(grad(loglik, argnums=(0, 1, 2, 3, 4)))

jax_loglik_batchscan, jax_grad_batchscan = make_loglik_and_grad(jax_sum_nll_batchscan)
jax_loglik_batchvmap, jax_grad_batchvmap = make_loglik_and_grad(jax_sum_nll_batchvmap)

# Warm up both
print("Warming up JAX JIT compilation...")
true_args = (TRUE_PARAMS['eta'], TRUE_PARAMS['kappa'],
             TRUE_PARAMS['a'], TRUE_PARAMS['b'], TRUE_PARAMS['x0'])
for name, ll_fn, g_fn in [("batchscan", jax_loglik_batchscan, jax_grad_batchscan),
                           ("batchvmap", jax_loglik_batchvmap, jax_grad_batchvmap)]:
    _ = ll_fn(*true_args)
    _ = g_fn(*true_args)
    ll_val = float(ll_fn(*true_args))
    print(f"  {name} log-likelihood at true params: {ll_val:.4f}")

print("Done.")

Warming up JAX JIT compilation...


/Users/afengler/Library/CloudStorage/OneDrive-Personal/proj_addm/efficient-fpt/src/efficient_fpt/jax/batch.py:301: UserWarning: Explicitly requested dtype float64 requested in full is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  jnp.full((batch_size, 1), a, dtype=dtype),


  batchscan log-likelihood at true params: -749.6751
  batchvmap log-likelihood at true params: -749.6751
Done.


## 3. Compare Likelihood Evaluation Speed

In [10]:
# Forward and gradient timings (warmed up; reports grad/fwd ratio)
def time_fn(fn, *args, n=20):
    """Median-of-3 batches of n calls. Returns seconds per call."""
    import statistics
    batches = []
    for _ in range(3):
        start = time.perf_counter()
        for _ in range(n):
            out = fn(*args)
            # Force device sync for JAX outputs.
            if hasattr(out, 'block_until_ready'):
                out.block_until_ready()
            elif isinstance(out, (list, tuple)):
                for x in out:
                    if hasattr(x, 'block_until_ready'):
                        x.block_until_ready()
        batches.append((time.perf_counter() - start) / n)
    return statistics.median(batches)

print("=" * 70)
print(f"Likelihood + gradient timing  (precision={'float64' if jax.config.jax_enable_x64 else 'float32'}, "
      f"TRUNC_NUM={TRUNC_NUM}, use_remat={JAX_USE_REMAT})")
print("=" * 70)

# Cython baseline
t_cython = time_fn(lambda: ll_cython(theta_true), n=20)
print(f"Cython:                {t_cython*1000:8.2f} ms/eval")
print()

# Pre-bind theta unpacking so the closure has minimal Python overhead.
def make_callers(ll_fn, g_fn):
    def call_ll():
        return ll_fn(theta_true[0], theta_true[1], theta_true[2], theta_true[3], theta_true[4])
    def call_grad():
        return g_fn(theta_true[0], theta_true[1], theta_true[2], theta_true[3], theta_true[4])
    return call_ll, call_grad

for name, ll_fn, g_fn in [("batchscan", jax_loglik_batchscan, jax_grad_batchscan),
                           ("batchvmap", jax_loglik_batchvmap, jax_grad_batchvmap)]:
    call_ll, call_grad = make_callers(ll_fn, g_fn)
    # Warm
    call_ll(); call_grad()
    t_fwd = time_fn(call_ll, n=20)
    t_grad = time_fn(call_grad, n=20)
    ratio = t_grad / t_fwd if t_fwd > 0 else float('nan')
    print(f"JAX {name}:")
    print(f"  forward:   {t_fwd*1000:8.2f} ms/eval")
    print(f"  gradient:  {t_grad*1000:8.2f} ms/eval   (grad/fwd = {ratio:.1f}x)")
    print()

print("Reference (from prior pymc_sampler_comparison run, float64+remat=False+TRUNC_NUM=10):")
print("  batchscan: forward 22.26 ms, gradient 921.06 ms (41x)")
print("  batchvmap: forward 36.68 ms, gradient 740.61 ms (20x)")
print("Healthy reverse-mode ratio is ~3-6x.")


Likelihood + gradient timing  (precision=float32, TRUNC_NUM=6, use_remat=True)
Cython:                    4.89 ms/eval

JAX batchscan:
  forward:      26.08 ms/eval
  gradient:    113.34 ms/eval   (grad/fwd = 4.3x)

JAX batchvmap:
  forward:      26.65 ms/eval
  gradient:    149.27 ms/eval   (grad/fwd = 5.6x)

Reference (from prior pymc_sampler_comparison run, float64+remat=False+TRUNC_NUM=10):
  batchscan: forward 22.26 ms, gradient 921.06 ms (41x)
  batchvmap: forward 36.68 ms, gradient 740.61 ms (20x)
Healthy reverse-mode ratio is ~3-6x.
